# Run On-Device LLM Inference with LiteRT-LM and Gemma 4

This tutorial demonstrates how to use the **LiteRT-LM** Python library to run efficient, on-device LLM inference using `.litertlm` model files.

[LiteRT-LM](https://ai.google.dev/edge/litert-lm) is a production-ready, open-source inference framework designed to deliver high-performance, cross-platform LLM deployments on edge devices.

* **Cross-Platform Support**: Run on Android, iOS, Web, Desktop, and IoT (e.g. Raspberry Pi).
* **Hardware Acceleration**: Get peak performance and system stability by leveraging GPU and NPU accelerators across diverse hardware.
* **Multi-Modality**: Build with LLMs that have vision and audio support.
* **Tool Use**: Function calling support for agentic workflows with constrained decoding for improved accuracy.
* **Broad Model Support**: Run Gemma, Llama, Phi-4, Qwen and more.

### Useful Links:
* **Official Documentation**: https://ai.google.dev/edge/litert-lm
* **GitHub Repository**: https://github.com/google-ai-edge/LiteRT-LM
* **Web Demo Page**: https://google-ai-edge.github.io/LiteRT-LM/web_demos/chat/index.html
* **LiteRT-LM Developers Blogpost**: https://developers.googleblog.com/blazing-fast-on-device-genai-with-litert-lm/

---

In this notebook, we will showcase the core capabilities of LiteRT-LM using the **Gemma 4 E2B** multimodal model in the following order:
1. **Basic text generation**
2. **Asynchronous streaming response**
3. **Multi-modality (Vision / Image inputs)**
4. **Multi-modality (Audio / Speech inputs)**
5. **Custom system instructions & conversation history** (with switchable compact personas)
6. **Speculative decoding with Multi-Token Prediction (MTP)** (optimized with streaming)
7. **Benchmarking model execution speeds**

## 1. Setup and Installation

First, let's install the required packages. We need `litert-lm-api` for the LiteRT-LM runtime, and `huggingface_hub` to easily download our optimized model from the Hugging Face model hub.


In [ ]:
!pip install -q litert-lm-api huggingface_hub

# Required for GPU
!apt-get update && apt-get install -y libvulkan1

## 2. Download the Gemma 4 E2B Model

The Gemma 4 E2B instruction-tuned model is hosted on Hugging Face in the `.litertlm` format, which is optimized specifically for on-device execution.

We will download the `gemma-4-E2B-it.litertlm` file from the [litert-community/gemma-4-E2B-it-litert-lm](https://huggingface.co/litert-community/gemma-4-E2B-it-litert-lm) repository.


In [ ]:
from huggingface_hub import hf_hub_download

print("Downloading Gemma 4 E2B model from Hugging Face. This may take a few minutes...")
model_path = hf_hub_download(
    repo_id="litert-community/gemma-4-E2B-it-litert-lm",
    filename="gemma-4-E2B-it.litertlm"
)
print(f"Downloaded model successfully to: {model_path}")

## 3. Basic Text Generation

To perform inference, we initialize the `Engine` with our downloaded model. The `Engine` manages model resources.

We then create a `Conversation` object, which handles conversation history and state. Using the `with` statement (context manager) ensures that all on-device memory and hardware resources are properly released when done.


In [ ]:
import litert_lm

# Load the model using the Engine. We will use Backend.CPU() for local CPU execution.
# (Note: GPU acceleration can be configured via backend=litert_lm.Backend.GPU() if supported)
with litert_lm.Engine(model_path, backend=litert_lm.Backend.CPU()) as engine:
  # Create a conversation instance
  with engine.create_conversation() as conversation:
    # Send a synchronous message
    response = conversation.send_message("What is the capital of France?")

    # Extract the response text
    text = response["content"][0]["text"]
    print(f"Response:\n{text}")

## 4. Asynchronous Streaming (Token-by-Token)

For interactive chat applications, waiting for the entire response to generate can feel slow.

LiteRT-LM provides `send_message_async`, which returns an iterator that yields response chunks in real-time as they are being decoded. This allows you to stream the output token-by-token.


In [ ]:
with litert_lm.Engine(model_path, backend=litert_lm.Backend.CPU()) as engine:
  with engine.create_conversation() as conversation:
    prompt = "Tell me a short 3-sentence story about a brave little robot."
    print(f"Prompt: {prompt}\n\nStreaming Response:\n", end="")

    # Start asynchronous streaming
    stream = conversation.send_message_async(prompt)
    for chunk in stream:
      # Response chunks are dictionary objects containing a content array
      for item in chunk.get("content", []):
        if item.get("type") == "text":
          print(item["text"], end="", flush=True)
    print()

## 5. Multi-Modality (Vision / Image Input)

The **Gemma 4 E2B** model natively supports vision (images) and audio inputs in addition to text.

To pass an image to the model:
1. Wrap the inputs in a `litert_lm.Contents` object.
2. Use `litert_lm.Content.ImageFile(image_path)` to specify the local path to the image.

*Note: While CPU execution is shown here for simplicity, offloading vision encoding to GPU (via `vision_backend=litert_lm.Backend.GPU()`) is strongly recommended for interactive use cases.*


In [ ]:
import urllib.request
from PIL import Image, ImageDraw
import os
from IPython.display import display

# Download a public image (a standard red STOP sign)
image_url = "https://upload.wikimedia.org/wikipedia/commons/f/f9/STOP_sign.jpg"
image_path = "stop_sign.jpg"

print(f"Downloading image from {image_url}...")
try:
  # Wikimedia requires a User-Agent header to allow downloads, otherwise it returns 403 Forbidden.
  req = urllib.request.Request(
      image_url,
      headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}
  )
  with urllib.request.urlopen(req) as response, open(image_path, 'wb') as out_file:
    out_file.write(response.read())
  print("Download complete.")
except Exception as e:
  print(f"Failed to download image: {e}")
  # Fallback: create a red square with text "STOP" if download fails
  img = Image.new("RGB", (300, 300), color="red")
  draw = ImageDraw.Draw(img)
  draw.text((120, 140), "STOP", fill="white")
  img.save(image_path)
  print("Created a fallback image.")

# Open and display the image
img = Image.open(image_path)
img.thumbnail((300, 300))
display(img)

# Load the model with vision support enabled on CPU
with litert_lm.Engine(
    model_path,
    backend=litert_lm.Backend.CPU(),
    vision_backend=litert_lm.Backend.CPU()  # Specify CPU backend for the vision processor
) as engine:
  with engine.create_conversation() as conversation:
    # Turn 1: Construct multimodal inputs combining image and a text prompt
    multimodal_input = litert_lm.Contents.of(
        litert_lm.Content.ImageFile(image_path),
        "Describe what you see in this image."
    )

    print("\nSending image + prompt to the model (streaming)...")
    stream = conversation.send_message_async(multimodal_input)
    print(f"\nModel Description:\n", end="")
    for chunk in stream:
      for item in chunk.get("content", []):
        if item.get("type") == "text":
          print(item["text"], end="", flush=True)
    print("\n\n" + "-" * 50 + "\n")

    # Turn 2: Ask the model to read the text (context and image are preserved!)
    print("Asking the model to perform OCR on the same image (streaming)...")
    stream2 = conversation.send_message_async("What text is written on the sign?")
    print(f"\nModel OCR Result:\n", end="")
    for chunk in stream2:
      for item in chunk.get("content", []):
        if item.get("type") == "text":
          print(item["text"], end="", flush=True)
    print()

# Clean up the temporary image
if os.path.exists(image_path):
  os.remove(image_path)

## 6. Multi-Modality (Audio / Speech Input)

In addition to images, **Gemma 4 E2B** natively supports audio inputs. This enables on-device **Automatic Speech Recognition (ASR)** and audio understanding.

In this section, we will:
1. Download a public audio sample (a WAV file containing spoken words).
2. Display an interactive audio player inside the notebook.
3. Send the audio along with a text prompt to perform on-device transcription (ASR) using streaming.

*Note: Similar to vision, offloading audio processing to CPU is shown here for simplicity, but hardware acceleration is recommended for production.*

In [ ]:
import urllib.request
from IPython.display import Audio, display
import os

# Download a public audio file (contains spoken words: "Have a wonderful day")
audio_url = "https://github.com/google-ai-edge/LiteRT-LM/raw/refs/heads/main/runtime/testdata/have_a_wonderful_day.wav"
audio_path = "have_a_wonderful_day.wav"

print(f"Downloading audio from {audio_url}...")
try:
  # Use a User-Agent to avoid potential 403 Forbidden errors
  req = urllib.request.Request(
      audio_url,
      headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
  )
  with urllib.request.urlopen(req) as response, open(audio_path, 'wb') as out_file:
    out_file.write(response.read())
  print("Download complete.")
except Exception as e:
  print(f"Failed to download audio: {e}")

# Play the audio in the notebook
if os.path.exists(audio_path):
  display(Audio(audio_path))

  # Load the model with audio support enabled on CPU
  with litert_lm.Engine(
      model_path,
      backend=litert_lm.Backend.CPU(),
      audio_backend=litert_lm.Backend.CPU()  # Specify CPU backend for the audio processor
  ) as engine:
    with engine.create_conversation() as conversation:
      # Construct multimodal inputs combining audio and a text prompt
      multimodal_input = litert_lm.Contents.of(
          litert_lm.Content.AudioFile(audio_path),
          "Transcribe this audio."
      )

      print("\nSending audio + prompt to the model (streaming ASR)...")
      stream = conversation.send_message_async(multimodal_input)
      print(f"\nModel Transcription:\n", end="")
      for chunk in stream:
        for item in chunk.get("content", []):
          if item.get("type") == "text":
            print(item["text"], end="", flush=True)
      print()

  # Clean up the temporary audio file
  os.remove(audio_path)
else:
  print("\nError: Audio file was not downloaded successfully. Skipping inference.")

## 7. System Instructions & Conversation History

A `Conversation` object preserves the state and history of your conversation.

You can also customize the assistant's persona and guidelines by passing a list of initial messages containing a **system instruction** when creating the conversation.

In this example, we provide two switchable options for the assistant's persona:
* **Option A: The Grumpy Pirate**: A curt, direct character who grunts and answers in at most 30 words.
* **Option B: The Wise Zen Master**: A calm, cryptic character who answers with a short riddle/koan of at most 30 words.

Both options are strictly constrained to at most 30 words. This demonstrates how system instructions can shape diverse assistant behaviors while keeping the generated output very brief to minimize on-device decoding latency.

In [ ]:
# Configure the system instruction. Choose one of the options below by uncommenting:

# Option A: The Grumpy Pirate (curt, direct)
assistant_name = "Pirate Assistant"
system_instruction = (
    "You are a grumpy, curt pirate who hates talking. You must always answer "
    "in a succinct but critical paragraph of at most 30 words, starting with a pirate grunt "
    "like 'Arr', 'Bah', or 'Avast'."
)

# # Option B: The Wise Zen Master (calm, cryptic) - Uncomment to switch:
# assistant_name = "Zen Master"
# system_instruction = (
#     "You are a wise, calm Zen Master. You must always answer with a short, "
#     "cryptic riddle or koan of at most 30 words that forces the user to reflect."
# )

initial_messages = [
    litert_lm.Message.system(system_instruction)
]

with litert_lm.Engine(model_path, backend=litert_lm.Backend.CPU()) as engine:
  # Initialize conversation with our custom system instruction
  with engine.create_conversation(messages=initial_messages) as conversation:

    # Turn 1 (Streaming)
    print(f"User: How can I write clean code?\n\n{assistant_name} (streaming):\n", end="")
    stream = conversation.send_message_async("How can I write clean code?")
    for chunk in stream:
      for item in chunk.get("content", []):
        if item.get("type") == "text":
          print(item["text"], end="", flush=True)
    print("\n\n" + "-" * 50 + "\n")

    # Turn 2 (Context is automatically maintained in this conversation, Streaming)
    print(f"User: And what about testing?\n\n{assistant_name} (streaming):\n", end="")
    stream2 = conversation.send_message_async("And what about testing?")
    for chunk in stream2:
      for item in chunk.get("content", []):
        if item.get("type") == "text":
          print(item["text"], end="", flush=True)
    print()

## 8. Multi-Token Prediction (MTP)

**Multi-Token Prediction (MTP)** is an advanced performance optimization in LiteRT-LM that significantly accelerates decoding speed. It works by predicting multiple tokens in parallel per execution step (speculative decoding).

To learn more about how MTP works and its performance benefits, check out the official [Google DeepMind Blog post](https://blog.google/innovation-and-ai/technology/developers-tools/multi-token-prediction-gemma-4/).

<img src="https://storage.googleapis.com/gweb-uniblog-publish-prod/images/Chart_Blog_Updated.width-1000.format-webp.webp" width="600" alt="MTP Speedup Chart" />

To use MTP, you simply set `enable_speculative_decoding=True` when creating the `Engine`.

In [ ]:
# Initialize with enable_speculative_decoding=True to leverage Multi-Token Prediction (MTP)
with litert_lm.Engine(
    model_path,
    backend=litert_lm.Backend.CPU(),
    enable_speculative_decoding=True
) as engine:
  with engine.create_conversation() as conversation:
    print("Sending prompt with MTP enabled (streaming):\n", end="")
    stream = conversation.send_message_async("Explain quantum computing in one sentence.")
    for chunk in stream:
      for item in chunk.get("content", []):
        if item.get("type") == "text":
          print(item["text"], end="", flush=True)
    print()

## 9. Benchmarking Performance

LiteRT-LM includes built-in benchmarking utilities that let you measure important performance metrics for on-device execution:
* **Model Init Time**: Time (in seconds) to load and prepare the model.
* **Time-to-First-Token (TTFT)**: Latency from sending the prompt to receiving the first generated token.
* **Prefill Speed**: Throughput during prompt ingestion (tokens/sec).
* **Decode Speed**: Generation speed of subsequent tokens (tokens/sec).

Let's measure these on our current environment.


In [ ]:
# Configure a benchmark run
benchmark = litert_lm.Benchmark(
    model_path,
    litert_lm.Backend.CPU(),
    prefill_tokens=64,  # Emulate a 64-token prompt
    decode_tokens=64,   # Emulate generating 64 tokens
)

print("Running benchmark. Please wait...")
results = benchmark.run()

print("\n=== Benchmark Results ===")
print(f"Model Init Time:           {results.init_time_in_second:.4f} seconds")
print(f"Time to First Token (TTFT): {results.time_to_first_token_in_second:.4f} seconds")
print(f"Prefill Speed:              {results.last_prefill_tokens_per_second:.2f} tokens/second")
print(f"Decode Speed:               {results.last_decode_tokens_per_second:.2f} tokens/second")

## Summary & Next Steps

Congratulations! You've completed this LiteRT-LM tutorial with Gemma 4 E2B. You now know how to:
1. Download `.litertlm` optimized model files from Hugging Face.
2. Run synchronous and asynchronous streaming text generation.
3. Use native on-device Multi-Modality by passing both **image** and **audio** files (running on-device OCR and ASR).
4. Configure custom system instructions with **switchable compact personas** and maintain conversation context.
5. Optimize performance with speculative decoding / Multi-Token Prediction (MTP).
6. Benchmark on-device execution speeds with the built-in suite.

For more details and native deployment platforms, visit:
* **Official Documentation**: [https://ai.google.dev/edge/litert-lm](https://ai.google.dev/edge/litert-lm)
* **GitHub Repository**: [https://github.com/google-ai-edge/LiteRT-LM](https://github.com/google-ai-edge/LiteRT-LM)